# Analyzing Potential Relationships between Permit Clusters and Economic Indicators

In this notebook, we undergo correlation tests to investigate potential relationships between correlation intensity (defined by a custom metric) and changes in affordability and accessibility for renters in Vancouver.

## Preliminaries

In [200]:
# Imports

# General

import numpy as np
import pandas as pd
import geopandas as gpd
import math
import json
import re
import string
from typing import Iterable


# Stats

from scipy.stats import pearsonr, spearmanr, kendalltau
from statsmodels.stats.multitest import multipletests

# Plotting

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook_connected' # For plotly graphs to render in this environment


In [201]:
# Set directory

PATH = "C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA"

In [202]:
# Load clustered builds

builds_df = pd.read_csv(f'{PATH}/CLUSTERED/builds_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(builds_df)

In [203]:
# Load clustered demos

demos_df = pd.read_csv(f'{PATH}/CLUSTERED/demos_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(demos_df)

In [204]:
# Load clustered renos

renos_df = pd.read_csv(f'{PATH}/CLUSTERED/renos_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(renos_df)

In [205]:
# Get unique neighborhoods across all permit sources

permit_nbhds = pd.concat([
    builds_df['nbhd'],
    renos_df['nbhd'],
    demos_df['nbhd']
]).str.lower().unique()   # capitalizes each neighborhood

permit_nbhds = set(permit_nbhds)

In [206]:
# Load economic data

econ_df = pd.read_csv(f'{PATH}/ECONOMIC/PROCESSED/full_economic_data.csv')


# examine_df(econ_df)

### Helper functions

In [207]:
# Function to examine dataframes

def examine_df(df,
               name = 'dataframe',
               include_stats = True,
               include_sample = True):
    
    """
    Check basic info about a dataframe df
    """
    
    print(f"\n\nNumber of records in the {name} is: {len(df)}\n")
    print(f"The columns in the {name} are: {df.columns}\n")
    print(f"\n Other info about {name}:\n")
    display(df.info())
    if include_stats == True:
        print(f'\n Basic statistical info about {name}:\n')
        display(df.describe())
    if include_sample == True:
        print(f"\n\nSample of records in the {name}:")
        display(df.head(5))

In [208]:
# Function to generate correlation heatmap

def gen_corr_heatmap(df):

    '''
    Generate correlation heatmap for numeric columns of a dataframe
    '''
    
    num_df = df.select_dtypes(include = 'number') # Restrict to numeric columns
    corr_matrix = num_df.corr() # Compute correlation matrix
    
    fig = px.imshow(
        corr_matrix,
        text_auto=True, # Include text
        color_continuous_scale='RdBu', # Set color scale
        aspect='auto', # Set aspect ratio
        title='Correlation Heatmap of Numeric Columns',
        zmin=-1,   # force range
        zmax=1
    ) # Generate heatmap figure
    fig.update_layout(title={'x': 0.5})
    
    fig.update_layout(width=1100, height=1100)          # bigger figure
    fig.update_xaxes(tickangle=45, tickfont=dict(size=9))
    fig.update_yaxes(tickfont=dict(size=9))
        
    fig.show(renderer="notebook")

In [209]:
# Function to prepare clustered permits for statistical testing

def prep_permits(df: pd.DataFrame, dev_type: str) -> pd.DataFrame:

    """
    Prepare cluster permit data for statistical testing
    """
    
    out = df.copy()
    
    out['index'] = np.arange(len(out))

    # Keep original index too (uncomment if you want a back-reference)
    # out['src_index'] = df.index.to_numpy()

    # Date → year
    out['issue_date'] = pd.to_datetime(out['issue_date'], errors='coerce')
    out['year'] = out['issue_date'].dt.year

    # Drop permits from 2025
    out = out[out['year'] <= 2024]

    # Dev type tag
    out['dev_type'] = dev_type  # e.g., {'build','reno','demo'}

    return out[['index','nbhd','year','cluster','dev_type']]

In [210]:
# Function to prepare economic data for statistical testing

def prep_econ(df: pd.DataFrame, valid_nbhds: set = permit_nbhds) -> pd.DataFrame:
    
    """
    Keep only neighborhood, zone, year, and change columns
    for correlation / regression analysis.
    """
    
    keep_cols = ['nbhd', 'zone', 'year'] + [c for c in df.columns if c.endswith('_change')]
    
    out = df[keep_cols].copy()
    
    # Filter to only neighborhoods that appear in permits
    out = out[out['nbhd'].isin(valid_nbhds)].reset_index(drop=True)
    
    return out

### Preparing Data

In [211]:
# Get prepared clustered builds dataframe

prep_builds_df = prep_permits(builds_df, 'build')

# examine_df(prep_builds_df)

In [212]:
# Get prepared clustered renos dataframe

prep_renos_df = prep_permits(renos_df, 'reno')

# examine_df(prep_renos_df)

In [213]:
# Get prepared clustered demos dataframe

prep_demos_df = prep_permits(demos_df, 'demo')

# examine_df(prep_demos_df)

In [214]:
# Concatenate different permit types

prep_permits_df = pd.concat(
    [prep_builds_df, prep_renos_df, prep_demos_df],
    axis=0,
    ignore_index=True
)

# Normalize neighborhood names
prep_permits_df['nbhd'] = prep_permits_df['nbhd'].str.lower()

# Add a globally unique UID
prep_permits_df['uid'] = (
    prep_permits_df['dev_type'].str[:1] + 
    prep_permits_df.index.astype(str)  # use the new concat index
)

# examine_df(prep_permits_df)

In [215]:
# Get prepared economic dataframe

prep_econ_df = prep_econ(econ_df)

# examine_df(prep_econ_df)

## Computing Cluster Scores and Lead Economic Metrics

In [216]:
# Function to compute cluster scores

def compute_cluster_scores(
    permits: pd.DataFrame,
    by_year: bool = True,
    smooth: float = 1.0) -> pd.DataFrame:
    
    """
    Compute neighborhood-by-year cluster scores:
      - share_{dev}_cK: within-(nbhd, year, dev_type) share of cluster K
      - nficf_{dev}_cK: share * log(1 / citywide_frequency_of_K) (with smoothing)

    We call the second score NF-ICF: Neighborhod Frequency - Inverse City Frequency
    
    """
    
    req = {'nbhd','year','dev_type','cluster'}

    # Check if required columns are present
    
    missing = req - set(permits.columns)  
    if missing:
        raise ValueError(f"Dataframe is missing required column(s): {missing}")

    df = permits.copy()

    # Neighborhood-year-dev_type-cluster counts
    g = (
        df.groupby(['nbhd','year','dev_type','cluster'])
          .size().rename('count').reset_index()
    )

    # Totals per neighborhood-year-dev_type
    tot = (
        g.groupby(['nbhd','year','dev_type'])['count']
         .sum().rename('total_devtype').reset_index()
    )
    
    out = g.merge(tot, on=['nbhd','year','dev_type'], how='left')

    # Shares 
    out['share'] = out['count'] / out['total_devtype'].replace({0: np.nan})

    # Citywide rarity weights 
    
    # Compute cluster counts at city level across all years.

    key_city = ['dev_type','cluster']
    key_total = ['dev_type']

    city_cluster = (
        df.groupby(key_city).size()
          .rename('city_cluster_count').reset_index()
    )
    
    city_total = (
        df.groupby(key_total).size()
          .rename('city_total_count').reset_index()
    )

    # Merge citywide counts back to out
    out = out.merge(city_cluster, on=key_city, how='left')
    out = out.merge(city_total, on=key_total, how='left')

    # IDF = log( (total + s) / (cluster + s) )  == log(1 / p_k) with smoothing
    out['idf'] = np.log((out['city_total_count'] + smooth) /
                        (out['city_cluster_count'] + smooth))

    # "TF–IDF" style score
    out['nficf'] = out['share'] * out['idf']

    # Wide pivot for easy merging with econ panel
    long_for_pivot = out.melt(
        id_vars=['nbhd','year','dev_type','cluster'],
        value_vars=['share','nficf'],
        var_name='score_type',
        value_name='score_value'
    )

    long_for_pivot['col'] = (
        long_for_pivot
        .apply(lambda r: f"{r['score_type']}_{r['dev_type']}_c{int(r['cluster'])}", axis=1)
    )

    wide = (
        long_for_pivot
        .pivot_table(index=['nbhd','year'], columns='col', values='score_value', aggfunc='first')
        .reset_index()
    )
    
    return wide


In [222]:
# Obtain all cluster scores by neighborhood/year

scores_df = compute_cluster_scores(prep_permits_df)

# examine_df(scores_df)

In [223]:
# Obtain dataframe with cluster scores and economic change metrics

dev_econ_df = prep_econ_df.merge(scores_df, on=["nbhd", "year"], how="left")

# examine_df(dev_econ_df)

In [224]:
# Function to add lead outcomes to dataframe

def add_lead_outcomes(df: pd.DataFrame,
                      group_col: str = "nbhd",
                      year_col: str = "year",
                      outcome_suffix: str = "_change",
                      leads: tuple[int, ...] = (1, 2, 3, 4, 5)) -> pd.DataFrame:
    
    """
    For each column ending with `outcome_suffix`, create lead-k columns by neighborhood:
      e.g., avg_rent_total_change -> avg_rent_total_change_lead1 ... lead5
    """
    
    out = df.sort_values([group_col, year_col]).copy()
    outcome_cols = [c for c in out.columns if c.endswith(outcome_suffix)]

    for c in outcome_cols:
        for k in leads:
            out[f"{c}_lead_{k}"] = out.groupby(group_col, sort=False)[c].shift(-k)

    return out

In [225]:
# Add lead outcomes

dev_econ_leads_df = add_lead_outcomes(dev_econ_df, leads=(1,2,3,4,5))

# examine_df(dev_econ_leads_df)

In [221]:
# Reality check for leads in a particular neighborhood

# Pick one neighborhood
nbhd_example = "mount pleasant"  

# A few metrics to inspect
metrics = ["avg_rent_total_change",
           "vacancy_rate_total_change",
           "med_rent_one_bedroom_change"]

# Gather the columns: base + lead1..lead5
cols_to_show = ["year"] + [
    col for m in metrics for col in [m] + [f"{m}_lead_{k}" for k in range(1,6)]
]

# Subset dataframe for that neighborhood
check_df = (
    dev_econ_leads_df
    .query("nbhd == @nbhd_example")
    .sort_values("year")[cols_to_show]
)

display(check_df.head(20))

,year,avg_rent_total_change,avg_rent_total_change_lead_1,avg_rent_total_change_lead_2,avg_rent_total_change_lead_3,avg_rent_total_change_lead_4,avg_rent_total_change_lead_5,vacancy_rate_total_change,vacancy_rate_total_change_lead_1,vacancy_rate_total_change_lead_2,vacancy_rate_total_change_lead_3,vacancy_rate_total_change_lead_4,vacancy_rate_total_change_lead_5,med_rent_one_bedroom_change,med_rent_one_bedroom_change_lead_1,med_rent_one_bedroom_change_lead_2,med_rent_one_bedroom_change_lead_3,med_rent_one_bedroom_change_lead_4,med_rent_one_bedroom_change_lead_5
104,2017,0.075078,0.040039,-0.004411,0.030839,0.004268,-0.014785,-0.1,-0.2,1.1,-0.1,-0.4,-0.4,0.063329,0.017930,0.046023,0.025762,0.053539,-0.012124
105,2018,0.040039,-0.004411,0.030839,0.004268,-0.014785,0.041392,-0.2,1.1,-0.1,-0.4,-0.4,-0.2,0.017930,0.046023,0.025762,0.053539,-0.012124,0.005085
106,2019,-0.004411,0.030839,0.004268,-0.014785,0.041392,0.038617,1.1,-0.1,-0.4,-0.4,-0.2,0.7,0.046023,0.025762,0.053539,-0.012124,0.005085,0.037894
107,2020,0.030839,0.004268,-0.014785,0.041392,0.038617,NaN,-0.1,-0.4,-0.4,-0.2,0.7,NaN,0.025762,0.053539,-0.012124,0.005085,0.037894,NaN
108,2021,0.004268,-0.014785,0.041392,0.038617,NaN,NaN,-0.4,-0.4,-0.2,0.7,NaN,NaN,0.053539,-0.012124,0.005085,0.037894,NaN,NaN
109,2022,-0.014785,0.041392,0.038617,NaN,NaN,NaN,-0.4,-0.2,0.7,NaN,NaN,NaN,-0.012124,0.005085,0.037894,NaN,NaN,NaN
110,2023,0.041392,0.038617,NaN,NaN,NaN,NaN,-0.2,0.7,NaN,NaN,NaN,NaN,0.005085,0.037894,NaN,NaN,NaN,NaN
111,2024,0.038617,NaN,NaN,NaN,NaN,NaN,0.7,NaN,NaN,NaN,NaN,NaN,0.037894,NaN,NaN,NaN,NaN,NaN


## Checking for Correlation

In [226]:
# Function to execute correlation tests
    
def corr_test(df: pd.DataFrame, x: str, y: str, method: str = "spearman") -> dict:
    """
    Compute correlation between two columns in df.
    Methods: 'pearson', 'spearman', 'kendall'.
    Returns a dict with correlation coefficient and p-value.
    """
    temp = df[[x, y]].dropna()
    if len(temp) < 5:  # skip tiny samples
        return {"x": x, "y": y, "n": len(temp),
                "method": method, "corr": None, "pval": None}
    
    if method == "pearson":
        corr, pval = pearsonr(temp[x], temp[y])
    elif method == "kendall":
        corr, pval = kendalltau(temp[x], temp[y])
    else:  # default to Spearman
        corr, pval = spearmanr(temp[x], temp[y])
    
    return {"x": x, "y": y, "n": len(temp),
            "method": method, "corr": corr, "pval": pval}

In [227]:
# Identify columns for testing

score_cols = [c for c in dev_econ_leads_df.columns
              if c.startswith("share_") or c.startswith("nficf_")]
econ_cols = [c for c in dev_econ_leads_df.columns
             if c.endswith("_change")]


In [228]:
# Function to apply FDR (false discovery rate) correction using Benjamini-Hochberg

def apply_fdr(df: pd.DataFrame,
              pval_col: str = "pval",
              alpha: float = 0.10,
              method: str = "fdr_bh") -> pd.DataFrame:
    
    """
    Apply multiple-testing correction (Benjamini–Hochberg by default).
    - Adds columns: 'pval_adj', 'significant'
    - Operates on rows where p-values are non-null; preserves others.
    """
    
    out = df.copy()
    mask = out[pval_col].notna()
    if mask.sum() == 0:
        out["pval_adj"] = pd.NA
        out["significant"] = False
        return out

    rej, p_adj, _, _ = multipletests(out.loc[mask, pval_col].values,
                                     alpha=alpha, method=method)
    out.loc[mask, "pval_adj"] = p_adj
    out.loc[mask, "significant"] = rej
    
    return out

In [229]:
# Function to check correlation for all pairs of cluster scores and economic metrics

def corr_matrix(df: pd.DataFrame,
                score_cols: list[str] = score_cols,
                econ_cols: list[str] = econ_cols,
                leads: Iterable[int] = (1,2,3,4,5),
                fdr_alpha: float = 0.10,
                fdr_method: str = "fdr_bh",
                method: str = "spearman"  # 'spearman' | 'pearson' | 'kendall'
               ) -> pd.DataFrame:
    """
    Run correlations for every (score_col, econ_col_lead_k) using the chosen method
    and apply FDR. Returns a DataFrame with raw and adjusted p-values.
    """
    rows = []
    for x in score_cols:
        for y in econ_cols:
            for k in leads:
                # tolerate both naming styles
                yk1 = f"{y}_lead{k}"
                yk2 = f"{y}_lead_{k}"
                yk = yk1 if yk1 in df.columns else (yk2 if yk2 in df.columns else None)
                if yk is None:
                    continue
                res = corr_test(df, x, yk, method=method)  # <-- pass method through
                res.update({"econ": y, "lead": k})
                rows.append(res)

    results = pd.DataFrame(rows)

    # Apply FDR across all tests (global BH)
    results = apply_fdr(results, pval_col="pval", alpha=fdr_alpha, method=fdr_method)

    # Nice ordering
    return (results
            .sort_values(["significant", "pval_adj", "pval"], ascending=[False, True, True])
            .reset_index(drop=True))

### Spearman's Rank Correlation Coefficient

In [230]:
# Run correlation tests

corr_results = corr_matrix(dev_econ_leads_df)

In [231]:
# Examine correlation results

examine_df(corr_results)



Number of records in the dataframe is: 2700

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2700 entries, 0 to 2699
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            2700 non-null   object 
 1   y            2700 non-null   object 
 2   n            2700 non-null   int64  
 3   method       2700 non-null   object 
 4   corr         2460 non-null   float64
 5   pval         2460 non-null   float64
 6   econ         2700 non-null   object 
 7   lead         2700 non-null   int64  
 8   pval_adj     2460 non-null   float64
 9   significant  2460 non-null   object 
dtypes: float64(3), int64(2), object(5)
memory usage: 211.1+ KB


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,2700.000000,2460.000000,2460.000000,2700.000000,2460.000000
mean,60.711111,-0.005376,0.464951,3.000000,0.838213
std,41.359066,0.196337,0.298775,1.414476,0.182923
min,0.000000,-0.942857,0.000106,1.000000,0.088298
25%,20.000000,-0.109624,0.201153,2.000000,0.787730
50%,59.000000,-0.003913,0.454329,3.000000,0.906019
75%,91.000000,0.097943,0.719190,4.000000,0.958401
max,174.000000,0.800000,1.000000,5.000000,1.000000




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,spearman,0.455723,0.000106,vacancy_rate_three_bedroom_plus_change,5,0.088298,True
1,share_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,spearman,0.455723,0.000106,vacancy_rate_three_bedroom_plus_change,5,0.088298,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,spearman,-0.383382,0.000192,avg_rent_one_bedroom_change,2,0.088298,True
3,share_build_c0,avg_rent_one_bedroom_change_lead_2,90,spearman,-0.383382,0.000192,avg_rent_one_bedroom_change,2,0.088298,True
4,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,spearman,-0.375065,0.000270,vacancy_rate_two_bedroom_change,2,0.088298,True


In [232]:
# Restrict to statistically significant correlations

# Adjusted significance threshold
alpha = 0.10  # try 0.10 for exploratory analysis

sig_results = corr_results.query("pval_adj < @alpha").copy()

# Filter for practical effect size AND NFICF variables only
sig_results = sig_results.loc[
    (sig_results['corr'].abs() >= 0.25) &
    (sig_results['x'].str.startswith("nficf"))
]

# Sort for readability
sig_results = sig_results.sort_values(
    ["pval_adj", "corr"], ascending=[True, False]
)

examine_df(sig_results)



Number of records in the dataframe is: 10

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 0 to 18
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            10 non-null     object 
 1   y            10 non-null     object 
 2   n            10 non-null     int64  
 3   method       10 non-null     object 
 4   corr         10 non-null     float64
 5   pval         10 non-null     float64
 6   econ         10 non-null     object 
 7   lead         10 non-null     int64  
 8   pval_adj     10 non-null     float64
 9   significant  10 non-null     object 
dtypes: float64(3), int64(2), object(5)
memory usage: 880.0+ bytes


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,10.000000,10.000000,10.000000,10.00000,10.000000
mean,74.600000,-0.124669,0.000441,2.90000,0.089171
std,16.971872,0.411141,0.000211,1.37032,0.001839
min,45.000000,-0.394015,0.000106,1.00000,0.088298
25%,69.000000,-0.388195,0.000293,2.00000,0.088298
50%,75.500000,-0.371476,0.000458,2.50000,0.088298
75%,90.000000,0.252797,0.000567,3.75000,0.088298
max,90.000000,0.486814,0.000753,5.00000,0.092659




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,spearman,0.455723,0.000106,vacancy_rate_three_bedroom_plus_change,5,0.088298,True
14,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_2,90,spearman,-0.355983,0.000574,vacancy_rate_three_bedroom_plus_change,2,0.088298,True
6,nficf_build_c0,vacancy_rate_total_change_lead_2,90,spearman,-0.367887,0.000361,vacancy_rate_total_change,2,0.088298,True
4,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,spearman,-0.375065,0.000270,vacancy_rate_two_bedroom_change,2,0.088298,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,spearman,-0.383382,0.000192,avg_rent_one_bedroom_change,2,0.088298,True


In [233]:
# Examine all statistically significant pairings

display(sig_results.head(50))

,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,spearman,0.455723,0.000106,vacancy_rate_three_bedroom_plus_change,5,0.088298,True
14,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_2,90,spearman,-0.355983,0.000574,vacancy_rate_three_bedroom_plus_change,2,0.088298,True
6,nficf_build_c0,vacancy_rate_total_change_lead_2,90,spearman,-0.367887,0.000361,vacancy_rate_total_change,2,0.088298,True
4,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,spearman,-0.375065,0.000270,vacancy_rate_two_bedroom_change,2,0.088298,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,spearman,-0.383382,0.000192,avg_rent_one_bedroom_change,2,0.088298,True
12,nficf_build_c0,vacancy_rate_one_bedroom_change_lead_3,75,spearman,-0.389800,0.000546,vacancy_rate_one_bedroom_change,3,0.088298,True
8,nficf_build_c2,vacancy_rate_three_bedroom_plus_change_lead_4,76,spearman,-0.392954,0.000446,vacancy_rate_three_bedroom_plus_change,4,0.088298,True
10,nficf_build_c0,vacancy_rate_total_change_lead_3,75,spearman,-0.394015,0.000469,vacancy_rate_total_change,3,0.088298,True
16,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_5,45,spearman,0.486814,0.000696,vacancy_rate_three_bedroom_plus_change,5,0.092659,True
18,nficf_build_c5,vacancy_rate_total_change_lead_1,48,spearman,0.469855,0.000753,vacancy_rate_total_change,1,0.092659,True


#### (Spearman) Correlation Findings Between Development Clusters and Economic Outcomes

Below are the statistically significant associations after FDR correction at 10% (|corr| ≥ 0.25).  
These results should be seen as **exploratory correlations**, not causal proof, but they provide suggestive patterns linking development activity to later rent and vacancy dynamics.

---

##### Renovations
- **NFICF_Reno Cluster 1 (Single-Detached Home Renovations)**  
  ↔ **Vacancy Rate (3+ bedroom units), Lead 5 years**  
  - Correlation: **+0.46** (n=67)  
  - Interpretation: Neighborhoods with higher relative intensity of **mainstream single-family renovations** appear to experience **higher vacancy rates in larger (3+ bedroom) rental units five years later**.  
  - *Speculative link:* Could indicate that widespread detached renovations gradually shift housing supply dynamics, leading to more turnover or increased availability in larger rental units over time.

---

##### New Buildings
- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Vacancy Rate (Total), Lead 2 years**  
  ↔ **Vacancy Rate (1BR, 2BR, 3+BR), Lead 2–3 years**  
  ↔ **Average Rent (1BR), Lead 2 years**  
  - Correlations: **Negative (−0.36 to −0.39)**  
  - Interpretation: Higher shares of **moderate-value detached/duplex housing** are linked to **lower vacancy rates and lower subsequent rent growth in 1BR units 2–3 years later**.  
  - *Speculative link:* Small-scale infill of detached/duplex units may relieve localized pressure on smaller rental markets, dampening rent increases and reducing churn in vacancy rates.

- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Vacancy Rate (3+ bedroom), Lead 5 years**  
  - Correlation: **+0.49** (n=45)  
  - Interpretation: Over longer horizons, the same detached/duplex activity correlates with **higher vacancy in larger rental units**.  
  - *Speculative link:* Detached/duplex building may compete with larger rentals for families, leading to more vacancies in the multi-bedroom rental stock over time.

- **NFICF_Build Cluster 2 (Mid-Value Family Housing)**  
  ↔ **Vacancy Rate (3+ bedroom), Lead 4 years**  
  - Correlation: **−0.39** (n=76)  
  - Interpretation: Neighborhoods with more **mid- to high-value family-oriented builds** later see **lower vacancy in larger rental units**.  
  - *Speculative link:* These family-focused builds may increase demand for larger rentals nearby (spillover effects), tightening vacancies in that segment.

- **NFICF_Build Cluster 5 (Duplexes with Secondary Suites)**  
  ↔ **Vacancy Rate (Total), Lead 1 year**  
  - Correlation: **+0.47** (n=48)  
  - Interpretation: A near-term link between **duplexes with many secondary suites** and **higher total vacancy rates**.  
  - *Speculative link:* Rapid introduction of secondary suites may expand rental options in the short run, temporarily raising vacancy.

---

##### Takeaways
- **Detached/duplex activity (Clusters 0 & 5)** shows **consistent associations with vacancy dynamics**, sometimes reducing pressure (lower vacancies/rents in smaller units after 2–3 years), but also raising vacancy in larger units after 5 years or overall vacancy after 1 year.  
- **Family-focused builds (Cluster 2)** may **tighten larger-unit rental markets**, suggesting competitive demand dynamics.  
- **Renovation activity in single-detached homes (Reno Cluster 1)** has a **long-run positive link to vacancies in large rental units**, hinting at gradual shifts in neighborhood composition.

These findings underline that **different forms of housing investment (renovation vs. new build, small-scale vs. large-scale) correlate with distinct lagged economic outcomes**, often with differing time horizons.


In [234]:
# Restrict to mildly statistically significant correlations (adjusted p < .2)

# Adjusted significance threshold
alpha = 0.20  # exploratory analysis

less_sig_results = corr_results.query("pval_adj < @alpha").copy()

# Filter for practical effect size AND NFICF variables only
less_sig_results = less_sig_results.loc[
    (less_sig_results['corr'].abs() >= 0.25) &
    (less_sig_results['x'].str.startswith("nficf"))
]

# Sort for readability
less_sig_results = less_sig_results.sort_values(
    ["pval_adj", "corr"], ascending=[True, False]
)

examine_df(less_sig_results)



Number of records in the dataframe is: 18

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
Index: 18 entries, 0 to 36
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            18 non-null     object 
 1   y            18 non-null     object 
 2   n            18 non-null     int64  
 3   method       18 non-null     object 
 4   corr         18 non-null     float64
 5   pval         18 non-null     float64
 6   econ         18 non-null     object 
 7   lead         18 non-null     int64  
 8   pval_adj     18 non-null     float64
 9   significant  18 non-null     object 
dtypes: float64(3), int64(2), object(5)
memory usage: 1.5+ KB


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,18.000000,18.000000,18.000000,18.000000,18.000000
mean,64.555556,-0.137321,0.001068,2.833333,0.113937
std,24.658186,0.441261,0.000775,1.382666,0.028978
min,12.000000,-0.797203,0.000106,1.000000,0.088298
25%,48.000000,-0.392165,0.000452,2.000000,0.088298
50%,71.000000,-0.361935,0.000725,2.500000,0.092659
75%,90.000000,0.429935,0.001642,4.000000,0.139823
max,90.000000,0.486814,0.002436,5.000000,0.157670




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,spearman,0.455723,0.000106,vacancy_rate_three_bedroom_plus_change,5,0.088298,True
14,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_2,90,spearman,-0.355983,0.000574,vacancy_rate_three_bedroom_plus_change,2,0.088298,True
6,nficf_build_c0,vacancy_rate_total_change_lead_2,90,spearman,-0.367887,0.000361,vacancy_rate_total_change,2,0.088298,True
4,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,spearman,-0.375065,0.000270,vacancy_rate_two_bedroom_change,2,0.088298,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,spearman,-0.383382,0.000192,avg_rent_one_bedroom_change,2,0.088298,True


In [235]:
# Examine all statistically significant pairings

display(less_sig_results.head(50))

,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,spearman,0.455723,0.000106,vacancy_rate_three_bedroom_plus_change,5,0.088298,True
14,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_2,90,spearman,-0.355983,0.000574,vacancy_rate_three_bedroom_plus_change,2,0.088298,True
6,nficf_build_c0,vacancy_rate_total_change_lead_2,90,spearman,-0.367887,0.000361,vacancy_rate_total_change,2,0.088298,True
4,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,spearman,-0.375065,0.000270,vacancy_rate_two_bedroom_change,2,0.088298,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,spearman,-0.383382,0.000192,avg_rent_one_bedroom_change,2,0.088298,True
12,nficf_build_c0,vacancy_rate_one_bedroom_change_lead_3,75,spearman,-0.389800,0.000546,vacancy_rate_one_bedroom_change,3,0.088298,True
8,nficf_build_c2,vacancy_rate_three_bedroom_plus_change_lead_4,76,spearman,-0.392954,0.000446,vacancy_rate_three_bedroom_plus_change,4,0.088298,True
10,nficf_build_c0,vacancy_rate_total_change_lead_3,75,spearman,-0.394015,0.000469,vacancy_rate_total_change,3,0.088298,True
16,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_5,45,spearman,0.486814,0.000696,vacancy_rate_three_bedroom_plus_change,5,0.092659,True
18,nficf_build_c5,vacancy_rate_total_change_lead_1,48,spearman,0.469855,0.000753,vacancy_rate_total_change,1,0.092659,True


#### Additional Exploratory (Spearman) Correlations (FDR-adjusted p < 0.20)

The following correlations did not meet the stricter 10% FDR threshold but are suggestive at the 20% level. They provide additional hypotheses worth keeping in mind, especially for exploratory or descriptive analysis.

---

##### New Buildings
- **NFICF_Build Cluster 5 (Duplexes with Secondary Suites)**  
  ↔ **Vacancy Rate (3+ bedroom), Lead 1 year**  
  - Correlation: **+0.45** (n=48)  
  - Interpretation: Duplexes with many secondary suites may temporarily **raise vacancy in larger rental units**, likely due to new suites competing directly with multi-bedroom rentals.

- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Median Rent (Studios), Lead 2 years**  
  - Correlation: **−0.33** (n=90)  
  - Interpretation: Higher detached/duplex activity is linked to **lower rent growth in studios after 2 years**, suggesting modest infill may dampen price pressures at the very small-unit end of the rental market.

- **NFICF_Build Cluster 7 (Large Multi-Unit Midrise Projects)**  
  ↔ **Average Rent (Total), Lead 4 years**  
  - Correlation: **−0.45** (n=48)  
  - Interpretation: Neighborhoods with more large multi-unit midrise projects see **slower overall rent growth after ~4 years**, potentially reflecting supply effects stabilizing rental prices.

- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Vacancy Rate (Total), Lead 4 years**  
  - Correlation: **+0.39** (n=60)  
  - Interpretation: Over longer horizons, detached/duplex activity may **contribute to higher overall vacancy rates**, possibly as new housing stock shifts neighborhood rental dynamics.

- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Vacancy Rate (Total), Lead 5 years**  
  - Correlation: **+0.44** (n=45)  
  - Interpretation: A similar effect as above, visible at 5 years, reinforcing the long-run link between detached/duplex activity and increased vacancies.

- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Vacancy Rate (Studios), Lead 2 years**  
  - Correlation: **−0.32** (n=90)  
  - Interpretation: In contrast, detached/duplex activity correlates with **lower vacancy in studio rentals after 2 years**, suggesting heterogeneous effects across unit types.

---

##### Renovations
- **NFICF_Reno Cluster 4 (Large-Scale Structural Repairs)**  
  ↔ **Median Rent (1BR), Lead 1 year**  
  - Correlation: **−0.62** (n=23)  
  - Interpretation: Rare, very high-value structural renovation projects are linked to **slower growth in median 1BR rents** the following year.  
  - *Speculative link:* Major building repairs may reduce market pressure temporarily (e.g., rehabs bringing units back online).

- **NFICF_Reno Cluster 5 (Multi-Unit Alterations)**  
  ↔ **Vacancy Rate (1BR), Lead 3 years**  
  - Correlation: **−0.80** (n=12)  
  - Interpretation: Strong negative association between **multi-unit alterations** and later 1BR vacancy, though based on a very small sample size.  
  - *Speculative link:* Could reflect targeted upgrades in buildings with persistently high occupancy.

---

##### Takeaways
- **Detached/duplex activity (Cluster 0)** shows **mixed effects**: lowering rents/vacancies in small-unit markets, while raising total or large-unit vacancies over longer horizons.  
- **Large multi-unit projects (Cluster 7)** may stabilize overall rents in the medium term.  
- **Renovation outliers (Clusters 4 & 5)** suggest some potential impact on rents and vacancies, but sample sizes are very small (n=12–23), so results should be interpreted with extreme caution.


### Pearson Correlation Coefficient

In [173]:
# Run correlation tests

pearson_corr_results = corr_matrix(dev_econ_leads_df, method="pearson")

# Examine correlation results

examine_df(pearson_corr_results)



Number of records in the dataframe is: 2700

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2700 entries, 0 to 2699
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            2700 non-null   object 
 1   y            2700 non-null   object 
 2   n            2700 non-null   int64  
 3   method       2700 non-null   object 
 4   corr         2460 non-null   float64
 5   pval         2460 non-null   float64
 6   econ         2700 non-null   object 
 7   lead         2700 non-null   int64  
 8   pval_adj     2460 non-null   float64
 9   significant  2460 non-null   object 
dtypes: float64(3), int64(2), object(5)
memory usage: 211.1+ KB


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,2700.000000,2460.000000,2460.000000,2700.000000,2460.000000
mean,60.711111,-0.000532,0.477921,3.000000,0.914075
std,41.359066,0.188700,0.286749,1.414476,0.090860
min,0.000000,-0.948431,0.000188,1.000000,0.230808
25%,20.000000,-0.099368,0.232504,2.000000,0.913381
50%,59.000000,-0.005260,0.470073,3.000000,0.922757
75%,91.000000,0.096122,0.717644,4.000000,0.954320
max,174.000000,0.822567,0.998627,5.000000,0.998627




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,pearson,-0.383882,0.000188,avg_rent_one_bedroom_change,2,0.230808,False
1,share_build_c0,avg_rent_one_bedroom_change_lead_2,90,pearson,-0.383882,0.000188,avg_rent_one_bedroom_change,2,0.230808,False
2,share_build_c4,med_rent_three_bedroom_plus_change_lead_1,43,pearson,-0.500496,0.000631,med_rent_three_bedroom_plus_change,1,0.383829,False
3,nficf_build_c4,med_rent_three_bedroom_plus_change_lead_1,43,pearson,-0.500496,0.000631,med_rent_three_bedroom_plus_change,1,0.383829,False
4,nficf_demo_c3,med_rent_one_bedroom_change_lead_1,105,pearson,-0.311361,0.001225,med_rent_one_bedroom_change,1,0.383829,False


In [174]:
# Restrict to statistically significant correlations

# Adjusted significance threshold
alpha = 0.10  # try 0.10 for exploratory analysis

pearson_sig_results = pearson_corr_results.query("pval_adj < @alpha").copy()

# Filter for practical effect size AND NFICF variables only
pearson_sig_results = pearson_sig_results.loc[
    (pearson_sig_results['corr'].abs() >= 0.25) &
    (pearson_sig_results['x'].str.startswith("nficf"))
]

# Sort for readability
pearson_sig_results = pearson_sig_results.sort_values(
    ["pval_adj", "corr"], ascending=[True, False]
)

examine_df(pearson_sig_results)



Number of records in the dataframe is: 0

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            0 non-null      object 
 1   y            0 non-null      object 
 2   n            0 non-null      int64  
 3   method       0 non-null      object 
 4   corr         0 non-null      float64
 5   pval         0 non-null      float64
 6   econ         0 non-null      object 
 7   lead         0 non-null      int64  
 8   pval_adj     0 non-null      float64
 9   significant  0 non-null      object 
dtypes: float64(3), int64(2), object(5)
memory usage: 0.0+ bytes


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant


In [177]:
# Restrict to mildly statistically significant correlations (adjusted p < .2)

# Adjusted significance threshold
alpha = 0.20  # exploratory analysis

pearson_less_sig_results = pearson_corr_results.query(".1 < pval_adj < @alpha").copy()

# Filter for practical effect size AND NFICF variables only
pearson_less_sig_results = pearson_less_sig_results.loc[
    (pearson_less_sig_results['corr'].abs() >= 0.25) &
    (pearson_less_sig_results['x'].str.startswith("nficf"))
]

# Sort for readability
pearson_less_sig_results = pearson_less_sig_results.sort_values(
    ["pval_adj", "corr"], ascending=[True, False]
)

examine_df(pearson_less_sig_results)



Number of records in the dataframe is: 0

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            0 non-null      object 
 1   y            0 non-null      object 
 2   n            0 non-null      int64  
 3   method       0 non-null      object 
 4   corr         0 non-null      float64
 5   pval         0 non-null      float64
 6   econ         0 non-null      object 
 7   lead         0 non-null      int64  
 8   pval_adj     0 non-null      float64
 9   significant  0 non-null      object 
dtypes: float64(3), int64(2), object(5)
memory usage: 0.0+ bytes


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant


**We conclude there is no apparent linear correlation detected by the Pearson Correlation Test.**

### Kendall Rank Correlation Coefficient

In [236]:
# Run correlation tests

kendall_corr_results = corr_matrix(dev_econ_leads_df, method="kendall")

# Examine correlation results

examine_df(kendall_corr_results)



Number of records in the dataframe is: 2700

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2700 entries, 0 to 2699
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            2700 non-null   object 
 1   y            2700 non-null   object 
 2   n            2700 non-null   int64  
 3   method       2700 non-null   object 
 4   corr         2460 non-null   float64
 5   pval         2460 non-null   float64
 6   econ         2700 non-null   object 
 7   lead         2700 non-null   int64  
 8   pval_adj     2460 non-null   float64
 9   significant  2460 non-null   object 
dtypes: float64(3), int64(2), object(5)
memory usage: 211.1+ KB


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,2700.000000,2460.000000,2460.000000,2700.000000,2460.000000
mean,60.711111,-0.004527,0.467591,3.000000,0.841304
std,41.359066,0.146835,0.302251,1.414476,0.180886
min,0.000000,-0.866667,0.000144,1.000000,0.091834
25%,20.000000,-0.075389,0.199359,2.000000,0.791985
50%,59.000000,-0.002000,0.456620,3.000000,0.910501
75%,91.000000,0.064395,0.735288,4.000000,0.974456
max,174.000000,0.670820,1.000000,5.000000,1.000000




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,kendall,0.343230,0.000144,vacancy_rate_three_bedroom_plus_change,5,0.091834,True
1,share_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,kendall,0.343230,0.000144,vacancy_rate_three_bedroom_plus_change,5,0.091834,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,kendall,-0.271966,0.000149,avg_rent_one_bedroom_change,2,0.091834,True
3,share_build_c0,avg_rent_one_bedroom_change_lead_2,90,kendall,-0.271966,0.000149,avg_rent_one_bedroom_change,2,0.091834,True
4,nficf_build_c0,vacancy_rate_total_change_lead_2,90,kendall,-0.260340,0.000315,vacancy_rate_total_change,2,0.096201,True


In [237]:
# Restrict to statistically significant correlations

# Adjusted significance threshold
alpha = 0.10  # try 0.10 for exploratory analysis

kendall_sig_results = kendall_corr_results.query("pval_adj < @alpha").copy()

# Filter for practical effect size AND NFICF variables only
kendall_sig_results = kendall_sig_results.loc[
    (kendall_sig_results['corr'].abs() >= 0.25) &
    (kendall_sig_results['x'].str.startswith("nficf"))
]

# Sort for readability
kendall_sig_results = kendall_sig_results.sort_values(
    ["pval_adj", "corr"], ascending=[True, False]
)

examine_df(kendall_sig_results)



Number of records in the dataframe is: 9

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
Index: 9 entries, 0 to 12
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            9 non-null      object 
 1   y            9 non-null      object 
 2   n            9 non-null      int64  
 3   method       9 non-null      object 
 4   corr         9 non-null      float64
 5   pval         9 non-null      float64
 6   econ         9 non-null      object 
 7   lead         9 non-null      int64  
 8   pval_adj     9 non-null      float64
 9   significant  9 non-null      object 
dtypes: float64(3), int64(2), object(5)
memory usage: 792.0+ bytes


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,9.000000,9.000000,9.000000,9.000000,9.000000
mean,77.555556,-0.130637,0.000432,3.111111,0.095230
std,15.025904,0.280940,0.000215,1.269296,0.001926
min,45.000000,-0.292371,0.000144,2.000000,0.091834
25%,75.000000,-0.271966,0.000315,2.000000,0.096201
50%,76.000000,-0.264795,0.000398,3.000000,0.096201
75%,90.000000,-0.260340,0.000597,4.000000,0.096201
max,90.000000,0.385185,0.000704,5.000000,0.096201




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,kendall,0.343230,0.000144,vacancy_rate_three_bedroom_plus_change,5,0.091834,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,kendall,-0.271966,0.000149,avg_rent_one_bedroom_change,2,0.091834,True
10,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_5,45,kendall,0.385185,0.000584,vacancy_rate_three_bedroom_plus_change,5,0.096201,True
4,nficf_build_c0,vacancy_rate_total_change_lead_2,90,kendall,-0.260340,0.000315,vacancy_rate_total_change,2,0.096201,True
6,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,kendall,-0.260692,0.000327,vacancy_rate_two_bedroom_change,2,0.096201,True


In [238]:
# Examine all statistically significant pairings

display(kendall_sig_results.head(50))

,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,kendall,0.343230,0.000144,vacancy_rate_three_bedroom_plus_change,5,0.091834,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,kendall,-0.271966,0.000149,avg_rent_one_bedroom_change,2,0.091834,True
10,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_5,45,kendall,0.385185,0.000584,vacancy_rate_three_bedroom_plus_change,5,0.096201,True
4,nficf_build_c0,vacancy_rate_total_change_lead_2,90,kendall,-0.260340,0.000315,vacancy_rate_total_change,2,0.096201,True
6,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,kendall,-0.260692,0.000327,vacancy_rate_two_bedroom_change,2,0.096201,True
16,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_2,90,kendall,-0.264795,0.000704,vacancy_rate_three_bedroom_plus_change,2,0.096201,True
14,nficf_build_c0,vacancy_rate_total_change_lead_3,75,kendall,-0.270385,0.000671,vacancy_rate_total_change,3,0.096201,True
8,nficf_build_c0,vacancy_rate_one_bedroom_change_lead_3,75,kendall,-0.283601,0.000398,vacancy_rate_one_bedroom_change,3,0.096201,True
12,nficf_build_c2,vacancy_rate_three_bedroom_plus_change_lead_4,76,kendall,-0.292371,0.000597,vacancy_rate_three_bedroom_plus_change,4,0.096201,True


#### (Kendall) Correlation Findings Between Development Clusters and Economic Outcomes

Below are the statistically significant associations after FDR correction at 10% (|corr| ≥ 0.25).  
These results should be seen as **exploratory correlations**, not causal proof, but they provide suggestive patterns linking development activity to later rent and vacancy dynamics.

---

##### Renovations
- **NFICF_Reno Cluster 1 (Single-Detached Home Renovations)**  
  ↔ **Vacancy Rate (3+ bedroom units), Lead 5 years**  
  - Correlation: **+0.34** (n=67)  
  - Interpretation: Renovation intensity in detached homes correlates with **higher vacancy in larger rentals after five years**.  
  - *Speculative link:* This may reflect longer-term neighborhood change, where detached home upgrades alter housing demand and lead to more vacancies in large rentals.

---

##### New Buildings
- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Average Rent (1BR), Lead 2 years**  
  ↔ **Vacancy Rate (Total, 1BR, 2BR, 3+BR), Lead 2–3 years**  
  - Correlations: **Negative (−0.26 to −0.28)**  
  - Interpretation: Higher levels of detached/duplex development are linked to **slower 1BR rent growth and reduced vacancy rates across multiple unit sizes 2–3 years later**.  
  - *Speculative link:* These small-scale builds may provide modest relief to local housing pressure, dampening rent increases and stabilizing occupancy.

- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Vacancy Rate (3+ bedroom units), Lead 5 years**  
  - Correlation: **+0.39** (n=45)  
  - Interpretation: Over the longer run, detached/duplex activity is associated with **higher vacancies in larger rental units**.  
  - *Speculative link:* Detached/duplex homes may substitute for larger rentals, drawing families out of the rental stock and leaving vacancies behind.

- **NFICF_Build Cluster 2 (Mid-Value Family Housing)**  
  ↔ **Vacancy Rate (3+ bedroom units), Lead 4 years**  
  - Correlation: **−0.29** (n=76)  
  - Interpretation: Family-oriented housing developments correlate with **lower vacancy rates in large rentals four years later**.  
  - *Speculative link:* May reflect demand spillovers, where the presence of mid-value family builds heightens overall attractiveness of neighborhoods for families.

---

##### Takeaways
- **Detached/duplex activity (Cluster 0)** again shows **dual effects**: short-term dampening of vacancies and rents, but longer-term increases in vacancies for large rentals.  
- **Family-oriented builds (Cluster 2)** show consistent tightening effects on larger rental unit vacancies.  
- **Renovation activity in detached homes (Reno Cluster 1)** maintains a long-run link to higher large-unit vacancies, echoing Spearman results.  

Overall, Kendall’s Tau results largely **reinforce the Spearman patterns**, lending robustness to the observed correlations while offering a more conservative measure better suited to smaller neighborhood-year samples.


In [239]:
# Restrict to mildly statistically significant correlations (adjusted p < .2)

# Adjusted significance threshold
alpha = 0.20  # exploratory analysis

kendall_less_sig_results = kendall_corr_results.query("pval_adj < @alpha").copy()

# Filter for practical effect size AND NFICF variables only
kendall_less_sig_results = kendall_less_sig_results.loc[
    (kendall_less_sig_results['corr'].abs() >= 0.25) &
    (kendall_less_sig_results['x'].str.startswith("nficf"))
]

# Sort for readability
kendall_less_sig_results = kendall_less_sig_results.sort_values(
    ["pval_adj", "corr"], ascending=[True, False]
)

examine_df(kendall_less_sig_results)



Number of records in the dataframe is: 16

The columns in the dataframe are: Index(['x', 'y', 'n', 'method', 'corr', 'pval', 'econ', 'lead', 'pval_adj',
       'significant'],
      dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
Index: 16 entries, 0 to 30
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   x            16 non-null     object 
 1   y            16 non-null     object 
 2   n            16 non-null     int64  
 3   method       16 non-null     object 
 4   corr         16 non-null     float64
 5   pval         16 non-null     float64
 6   econ         16 non-null     object 
 7   lead         16 non-null     int64  
 8   pval_adj     16 non-null     float64
 9   significant  16 non-null     object 
dtypes: float64(3), int64(2), object(5)
memory usage: 1.4+ KB


None


 Basic statistical info about dataframe:



,n,corr,pval,lead,pval_adj
count,16.000000,16.000000,16.000000,16.000000,16.000000
mean,64.875000,-0.013207,0.001210,2.812500,0.128106
std,18.778978,0.313849,0.001087,1.515201,0.048071
min,43.000000,-0.318445,0.000144,1.000000,0.091834
25%,48.000000,-0.274875,0.000381,1.750000,0.096201
50%,63.500000,-0.260516,0.000687,2.500000,0.096201
75%,79.500000,0.330982,0.002376,4.000000,0.196578
max,90.000000,0.385185,0.002885,5.000000,0.197143




Sample of records in the dataframe:


,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,kendall,0.343230,0.000144,vacancy_rate_three_bedroom_plus_change,5,0.091834,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,kendall,-0.271966,0.000149,avg_rent_one_bedroom_change,2,0.091834,True
10,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_5,45,kendall,0.385185,0.000584,vacancy_rate_three_bedroom_plus_change,5,0.096201,True
4,nficf_build_c0,vacancy_rate_total_change_lead_2,90,kendall,-0.260340,0.000315,vacancy_rate_total_change,2,0.096201,True
6,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,kendall,-0.260692,0.000327,vacancy_rate_two_bedroom_change,2,0.096201,True


In [240]:
# Examine all statistically significant pairings

display(kendall_less_sig_results.head(50))

,x,y,n,method,corr,pval,econ,lead,pval_adj,significant
0,nficf_reno_c1,vacancy_rate_three_bedroom_plus_change_lead_5,67,kendall,0.343230,0.000144,vacancy_rate_three_bedroom_plus_change,5,0.091834,True
2,nficf_build_c0,avg_rent_one_bedroom_change_lead_2,90,kendall,-0.271966,0.000149,avg_rent_one_bedroom_change,2,0.091834,True
10,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_5,45,kendall,0.385185,0.000584,vacancy_rate_three_bedroom_plus_change,5,0.096201,True
4,nficf_build_c0,vacancy_rate_total_change_lead_2,90,kendall,-0.260340,0.000315,vacancy_rate_total_change,2,0.096201,True
6,nficf_build_c0,vacancy_rate_two_bedroom_change_lead_2,90,kendall,-0.260692,0.000327,vacancy_rate_two_bedroom_change,2,0.096201,True
16,nficf_build_c0,vacancy_rate_three_bedroom_plus_change_lead_2,90,kendall,-0.264795,0.000704,vacancy_rate_three_bedroom_plus_change,2,0.096201,True
14,nficf_build_c0,vacancy_rate_total_change_lead_3,75,kendall,-0.270385,0.000671,vacancy_rate_total_change,3,0.096201,True
8,nficf_build_c0,vacancy_rate_one_bedroom_change_lead_3,75,kendall,-0.283601,0.000398,vacancy_rate_one_bedroom_change,3,0.096201,True
12,nficf_build_c2,vacancy_rate_three_bedroom_plus_change_lead_4,76,kendall,-0.292371,0.000597,vacancy_rate_three_bedroom_plus_change,4,0.096201,True
18,nficf_build_c0,vacancy_rate_total_change_lead_5,45,kendall,0.346702,0.000923,vacancy_rate_total_change,5,0.103830,False


#### Additional Exploratory (Kendall) Correlations (FDR-adjusted p < 0.20)

The following correlations did not meet the stricter 10% FDR threshold but are suggestive at the 20% level. They provide additional hypotheses worth keeping in mind, especially for exploratory or descriptive analysis.

---

##### New Buildings
- **NFICF_Build Cluster 0 (Small Detached & Duplex Mix)**  
  ↔ **Vacancy Rate (Total), Lead 4–5 years**  
  - Correlations: **+0.27 to +0.35** (n=45–60)  
  - Interpretation: Detached/duplex activity shows a **longer-run positive association with higher overall vacancy rates**, suggesting that new detached/duplex stock may eventually compete with existing rentals and loosen occupancy.

- **NFICF_Build Cluster 5 (Duplexes with Secondary Suites)**  
  ↔ **Vacancy Rate (Total), Lead 1 year**  
  ↔ **Vacancy Rate (3+ BR), Lead 1 year**  
  ↔ **Vacancy Rate (Studios), Lead 1 year**  
  - Correlations: **+0.30 to +0.33** (n≈48)  
  - Interpretation: Duplex + suite construction is associated with **higher vacancies across multiple rental segments within a year**, possibly due to rapid new suite supply.  
  - *Speculative link:* New secondary suites might immediately expand rental options, temporarily raising vacancy across the board.

- **NFICF_Build Cluster 7 (Large Multi-Unit Midrise Projects)**  
  ↔ **Average Rent (Total), Lead 4 years**  
  - Correlation: **−0.30** (n=48)  
  - Interpretation: Midrise multi-unit projects correlate with **slower growth in overall rents after several years**, consistent with a stabilizing supply effect.

- **NFICF_Build Cluster 4 (Small Dwellings with Suites/Laneways)**  
  ↔ **Median Rent (3+ BR), Lead 1 year**  
  - Correlation: **−0.32** (n=43)  
  - Interpretation: Suite- or laneway-linked small projects are associated with **reduced rent growth in larger units** the following year.  
  - *Speculative link:* May reflect local substitution effects, where added suites reduce pressure on family-sized rentals.

---

##### Takeaways
- **Detached/duplex clusters (0 & 5)** show **short-run increases in vacancy** (Cluster 5) and **long-run increases in total vacancies** (Cluster 0), suggesting different temporal dynamics by cluster type.  
- **Large multi-unit projects (Cluster 7)** again appear to **moderate rent growth**, reinforcing Spearman results.  
- **Laneway/suite-focused builds (Cluster 4)** may have a dampening effect on rent growth in larger units, hinting at subtle substitution dynamics.  

Overall, these Kendall results at the 20% level support the Spearman findings while highlighting **short-term vacancy increases from suite-heavy duplexes** and **long-term loosening of vacancy from detached/duplex infill**.


## Visualizing Correlation

In [243]:
# Function to plot correlation bar chart

def plot_corr_bars(df: pd.DataFrame, top_n: int = 40, title: str = "Correlations (adj p < 0.20)"):
    
    """
    Plot a colored bar chart of correlations for (x, econ, lead) pairs.
    - df: expected to already be filtered to pval_adj < 0.20 (e.g., less_sig_results)
    - top_n: show the top N by absolute correlation for readability
    """

    # Basic safety filter
    d = df.copy()
    d = d.dropna(subset=["corr", "pval_adj"])
    
    # Build a compact label: "x → econ (Lk)"
    # If your y column already has the full "econ_leadK" name, you can use that instead.
    d["pair"] = d.apply(lambda r: f"{r['x']} → {r['econ']} (L{int(r['lead'])})", axis=1)

    # Sort by absolute correlation and keep top N for readability
    d = d.sort_values("corr", key=lambda s: s.abs(), ascending=False).head(top_n)

    # Make the x-axis order match the sorted order
    d["pair"] = pd.Categorical(d["pair"], categories=d["pair"], ordered=True)

    # Simple diverging bar chart
    fig = px.bar(
        d,
        x="pair",
        y="corr",
        color="corr",
        color_continuous_scale="RdBu",  # blue = negative, red = positive
        hover_data={
            "pair": True,
            "corr": ':.3f',
            "pval_adj": ':.3f',
            "pval": ':.3f',
            "n": True,
            "method": True,
            "lead": True,
            "x": False, "y": False, "econ": False  # already encoded in 'pair'
        },
        title=title
    )

    # Style tweaks
    fig.update_layout(
        xaxis_title="Cluster score → Economic metric (Lead)",
        yaxis_title="Correlation",
        coloraxis_colorbar_title="corr",
        bargap=0.2,
        height=600,
        margin=dict(l=40, r=20, t=60, b=120)
    )
    fig.update_xaxes(tickangle=45)

    # Zero line
    fig.add_hline(y=0, line_width=1, line_color="gray", opacity=0.6)

    return fig

### Spearman's Rank Correlation Coefficient

In [244]:
# Check suggestive correlations ( adj p < .2 ) for Spearman testing

fig = plot_corr_bars(less_sig_results, top_n=40, title="Spearman Correlations (adj p < 0.20)")
fig.show(renderer = 'notebook_connected')

In [248]:
# Save results

less_sig_results.to_csv(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\data\spearman.csv")

### Kendall Rank Correlation Coefficient

In [245]:
# Check suggestive correlations ( adj p < .2 ) for Spearman testing

fig = plot_corr_bars(kendall_less_sig_results, top_n=40, title="Kendall Correlations (adj p < 0.20)")
fig.show(renderer = 'notebook_connected')

In [249]:
# Save results

kendall_less_sig_results.to_csv(r"C:\Users\emshe\Desktop\BRAINSTATION\CAPSTONE\GIT_REPO\DEMO\data\kendall.csv")